# 01 · Coleta dos dados — SCR.data (Banco Central do Brasil)

**Projeto:** Carteira de crédito e inadimplência no Sistema Financeiro Nacional
**Etapa:** coleta e reconhecimento dos dados brutos

Este notebook baixa os dados mensais do SCR.data e documenta sua estrutura antes do tratamento.

| | |
|---|---|
| **Fonte** | [SCR.data — Portal de Dados Abertos do BCB](https://dadosabertos.bcb.gov.br/dataset/scr_data) |
| **Conteúdo** | Operações de crédito agregadas por UF, segmento da instituição, tipo de cliente, porte, modalidade, origem dos recursos e indexador |
| **Formato** | Um arquivo ZIP por ano, com um CSV mensal separado por `;` |
| **Licença** | Open Data Commons Open Database License (ODbL) |

Os arquivos brutos são gravados em `data/raw/` e não são versionados (ver `.gitignore`); este notebook os reproduz.

In [ ]:
import zipfile
from pathlib import Path

import pandas as pd
import requests

RAW = Path("../data/raw")
RAW.mkdir(parents=True, exist_ok=True)

URL = "https://www.bcb.gov.br/pda/desig/scrdata_{ano}.zip"
ANOS = [2023, 2024, 2025, 2026]

## 1. Download

O download é feito em blocos e gravado em arquivo temporário (`.part`), renomeado apenas ao final. Downloads interrompidos não são confundidos com arquivos completos, e anos já baixados são ignorados, o que torna a execução repetível.

In [ ]:
def baixar(ano):
    """Baixa o ZIP anual do SCR.data, se ainda não existir localmente."""
    destino = RAW / f"scrdata_{ano}.zip"
    if destino.exists():
        print(f"{destino.name} já existe ({destino.stat().st_size / 1e6:.1f} MB)")
        return destino

    temporario = destino.with_suffix(".zip.part")
    with requests.get(URL.format(ano=ano), stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(temporario, "wb") as f:
            for bloco in r.iter_content(chunk_size=1024 * 1024):
                f.write(bloco)
    temporario.rename(destino)
    print(f"{destino.name} baixado ({destino.stat().st_size / 1e6:.1f} MB)")
    return destino


for ano in ANOS:
    try:
        baixar(ano)
    except requests.HTTPError as erro:
        print(f"Falha no download de {ano}: {erro}")

## 2. Inventário dos arquivos

Arquivos mensais disponíveis e volume descompactado.

In [ ]:
inventario = []
for zip_path in sorted(RAW.glob("scrdata_*.zip")):
    with zipfile.ZipFile(zip_path) as z:
        for info in z.infolist():
            inventario.append({
                "zip": zip_path.name,
                "arquivo": info.filename,
                "tamanho_mb": round(info.file_size / 1e6, 1),
            })

inventario = pd.DataFrame(inventario)
print(f"{len(inventario)} arquivos, {inventario['tamanho_mb'].sum():,.0f} MB descompactados")
inventario

## 3. Estrutura dos dados

Leitura de uma amostra de 1.000 linhas do mês mais recente, direto do ZIP, para identificar colunas, tipos e formato numérico. A codificação não é documentada pela fonte; a leitura tenta UTF-8 e, em caso de falha, Latin-1.

In [ ]:
def ler_amostra(zip_path, nome_arquivo, linhas=1000):
    """Lê as primeiras linhas de um CSV dentro do ZIP, testando a codificação."""
    for encoding in ["utf-8", "latin-1"]:
        try:
            with zipfile.ZipFile(zip_path) as z, z.open(nome_arquivo) as f:
                return pd.read_csv(f, sep=";", nrows=linhas, encoding=encoding, decimal=",")
        except UnicodeDecodeError:
            continue
    raise ValueError(f"Não foi possível ler {nome_arquivo} em UTF-8 nem Latin-1")


ultimo = inventario.sort_values("arquivo").iloc[-1]
amostra = ler_amostra(RAW / ultimo["zip"], ultimo["arquivo"])
print("Mês mais recente:", ultimo["arquivo"])
amostra.head()

In [ ]:
pd.DataFrame({
    "tipo": amostra.dtypes.astype(str),
    "exemplo": amostra.iloc[0],
    "valores_distintos_na_amostra": amostra.nunique(),
})

## Principais observações

- **Cobertura:** 45 arquivos mensais, de jan/2023 a set/2026, somando cerca de 4,5 GB descompactados. O volume exige processamento mês a mês no tratamento.
- **Estrutura:** cada linha é uma combinação de recortes (data, UF, segmento, cliente, CNAE/ocupação, porte, modalidade, submodalidade, origem e indexador) com os saldos correspondentes: carteira a vencer por prazo, vencidos, carteira ativa, carteira inadimplente e ativo problemático.
- **Formato numérico:** os valores usam vírgula decimal e são lidos corretamente como números com `decimal=","`.
- **Ponto de atenção:** `numero_de_operacoes` traz o valor `-1` em parte das linhas, o que impede somá-lo diretamente. A coluna é tratada no notebook 02.

**Próxima etapa:** [`02_tratamento.ipynb`](02_tratamento.ipynb) — seleção de recortes, agregação e geração da base analítica.